# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [5]:
#!pip install pandas

In [4]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [45]:
df.head(6)

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...
5,L'Or Espresso Café,Je trouve le goût médiocre. La mousse ne tient...


## LLMChain

In [ ]:
!pip install langchain_community

In [7]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [20]:
#Replace None by your own value and justify
llm = ChatOpenAI(model="gpt-6-luna", temperature=1)
#  I chose a low temperature to make responses creative and less predictable while testing the chains

In [27]:
prompt = ChatPromptTemplate.from_template("Describe this product in 1-2 statements: {name}")

In [22]:
chain = prompt | llm

In [28]:
product = "Airfryer"
chain.invoke({"name":product})

AIMessage(content='An air fryer is a countertop kitchen appliance that cooks food by circulating hot air around it. It can make foods such as fries, vegetables, and chicken crisp with little or no added oil. Most models have a removable basket, adjustable temperature and timer controls, and are easy to use for quick meals and reheating leftovers.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 13, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 34, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EdFeqZOkjtk3Z8CjN4uyDNDU2t8Ad', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1788a-383c-70ea-b493-5bc3de7857ba-0', usage_metadata={'input_tokens': 13, 'output_tokens':

## SimpleSequentialChain

In [24]:
from langchain.chains import SimpleSequentialChain

In [34]:
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Describe this product in 1-2 statements: {name}"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [35]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "What is the avg price for this product?: {product}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [36]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [37]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
An air fryer cooks food by circulating hot air, creating a crisp exterior with little to no oil. It’s a convenient alternative to deep-frying for foods like fries, vegetables, and chicken.
A typical air fryer costs about **$100** in the U.S. Prices commonly range from **$50 to $200**, depending on capacity and features.

> Finished chain.


'A typical air fryer costs about **$100** in the U.S. Prices commonly range from **$50 to $200**, depending on capacity and features.'

**Repeat the above twice for different products**

In [38]:
overall_simple_chain.run("iphone")



> Entering new SimpleSequentialChain chain...
The iPhone is Apple’s smartphone, combining a touchscreen, camera, and internet access in one device. It runs iOS and supports apps for communication, work, entertainment, and more.
A new iPhone typically costs about **$800–$1,000 USD** on average, depending on the model and storage. Older or refurbished models can cost considerably less.

> Finished chain.


'A new iPhone typically costs about **$800–$1,000 USD** on average, depending on the model and storage. Older or refurbished models can cost considerably less.'

In [39]:
overall_simple_chain.run("fritzcola")



> Entering new SimpleSequentialChain chain...
fritz-kola is a German cola brand known for its bold taste and high caffeine content. It also offers sugar-free and other soft-drink varieties.
A 330 ml bottle of fritz-kola typically costs about **€1.50 on average** in Germany, excluding the bottle deposit. Prices vary by retailer and location.

> Finished chain.


'A 330 ml bottle of fritz-kola typically costs about **€1.50 on average** in Germany, excluding the bottle deposit. Prices vary by retailer and location.'

## SequentialChain

In [40]:
from langchain.chains import SequentialChain

In [53]:
llm = ChatOpenAI(model="gpt-6-luna",temperature=1)


first_prompt = ChatPromptTemplate.from_template(
  "Translate this text into English: {text}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_version"
                    )


In [54]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this review in 2 sentences: {review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="Summary"
                    )


In [55]:
third_prompt = ChatPromptTemplate.from_template(
   "Translate this summary into Arabic: {summary}"
)
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="Arabic_version"
                      )


In [56]:
fourth_prompt = ChatPromptTemplate.from_template(
        "Is this review negative, positive, or neutral?: {review}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="Sentiment"
                     )


In [58]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["text"],
    output_variables=[
        "English_version",
        "Summary",
        "Arabic_version",
        "Sentiment",
    ],
    verbose=True,
)

ValidationError: 1 validation error for SequentialChain
  Value error, Missing required input keys: {'review'}, only had {'text', 'English_version'} [type=value_error, input_value={'chains': [LLMChain(verb...ment'], 'verbose': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error

In [ ]:
review = df.Review[5]
overall_chain(review)

In [ ]:
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["text"],
    output_variables=[
        "English_version",
        "Summary",
        "Arabic_version",
        "Sentiment",
    ],
    verbose=True,
)

**Repeat the above twice for different products or reviews**

## Router Chain

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [ ]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [ ]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(temperature=0)

In [ ]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [ ]:
chain.run("What is black body radiation?")

In [ ]:
chain.run("what is 2 + 2")

In [ ]:
chain.run("Why does every cell in our body contain DNA?")

**Repeat the above at least once for different inputs and chains executions - Be creative!**